# 🩺 Breast Cancer Data Analysis & Prediction
### Using Machine Learning on the Wisconsin Breast Cancer Dataset
---
**Dataset:** [Breast Cancer Wisconsin (Diagnostic) - Kaggle](https://www.kaggle.com/datasets/yasserh/breast-cancer-dataset)  
**Goal:** Classify tumors as Malignant (M) or Benign (B) using machine learning models  
**Libraries:** pandas, numpy, matplotlib, seaborn, scikit-learn, plotly, joblib

## 📦 1. Import Libraries

In [1]:
# Windows asyncio fix for Plotly/Jupyter
import sys, asyncio
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

# Core Libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization Libraries
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for nbconvert
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Scikit-learn - Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
)
from sklearn.decomposition import PCA

# Scikit-learn - Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Scikit-learn - Metrics
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

# Model Persistence
import joblib

# Styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
PALETTE = {'M': '#E74C3C', 'B': '#2ECC71'}

print('✅ All libraries imported successfully!')
print(f'   pandas  {pd.__version__}  |  numpy {np.__version__}  |  sklearn {__import__("sklearn").__version__}')

✅ All libraries imported successfully!
   pandas  3.0.2  |  numpy 2.4.4  |  sklearn 1.9.1


## 📂 2. Load & Inspect Dataset

In [2]:
# Load the dataset
df = pd.read_csv('Breast_cancer_dataset.csv')

print(f'Dataset Shape: {df.shape}')
print(f'Rows: {df.shape[0]} | Columns: {df.shape[1]}')
print('\n--- First 5 Rows ---')
df.head()

Dataset Shape: (569, 32)
Rows: 569 | Columns: 32

--- First 5 Rows ---


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842303,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,842304,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,842305,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,842306,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [3]:
# Dataset info
print('--- Dataset Info ---')
df.info()

--- Dataset Info ---
<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se    

In [4]:
# Statistical summary
print('--- Statistical Summary ---')
df.describe().round(3)

--- Statistical Summary ---


,id,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
count,569.0,569.000,569.000,569.000,569.000,569.000,569.000,569.000,569.000,569.000,...,569.000,569.000,569.000,569.000,569.000,569.000,569.000,569.000,569.000,569.000
mean,842586.0,14.127,19.290,91.969,654.889,0.096,0.104,0.089,0.049,0.181,...,16.269,25.677,107.261,880.583,0.132,0.254,0.272,0.115,0.290,0.084
std,164.4,3.524,4.301,24.299,351.914,0.014,0.053,0.080,0.039,0.027,...,4.833,6.146,33.603,569.357,0.023,0.157,0.209,0.066,0.062,0.018
min,842302.0,6.981,9.710,43.790,143.500,0.053,0.019,0.000,0.000,0.106,...,7.930,12.020,50.410,185.200,0.071,0.027,0.000,0.000,0.156,0.055
25%,842444.0,11.700,16.170,75.170,420.300,0.086,0.065,0.030,0.020,0.162,...,13.010,21.080,84.110,515.300,0.117,0.147,0.114,0.065,0.250,0.071
50%,842586.0,13.370,18.840,86.240,551.100,0.096,0.093,0.062,0.034,0.179,...,14.970,25.410,97.660,686.500,0.131,0.212,0.227,0.100,0.282,0.080
75%,842728.0,15.780,21.800,104.100,782.700,0.105,0.130,0.131,0.074,0.196,...,18.790,29.720,125.400,1084.000,0.146,0.339,0.383,0.161,0.318,0.092
max,842870.0,28.110,39.280,188.500,2501.000,0.163,0.345,0.427,0.201,0.304,...,36.040,49.540,251.200,4254.000,0.223,1.058,1.252,0.291,0.664,0.208


## 🔍 3. Data Preprocessing & Cleaning

In [5]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing,
    'Percentage (%)': missing_pct.round(2)
})
missing_df = missing_df[missing_df['Missing Values'] > 0]

if missing_df.empty:
    print('✅ No missing values found in the dataset!')
else:
    print('⚠️  Missing values detected:')
    print(missing_df)

✅ No missing values found in the dataset!


In [6]:
# Drop 'id' column (not useful for prediction)
if 'id' in df.columns:
    df.drop(columns=['id'], inplace=True)
    print('✅ Dropped "id" column')

# Drop unnamed/empty columns
unnamed_cols = [col for col in df.columns if 'Unnamed' in str(col)]
if unnamed_cols:
    df.drop(columns=unnamed_cols, inplace=True)
    print(f'✅ Dropped unnamed columns: {unnamed_cols}')

# Check for duplicates
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print(f'✅ Removed {duplicates} duplicate rows')

print(f'\nFinal dataset shape: {df.shape}')

✅ Dropped "id" column
Duplicate rows: 0

Final dataset shape: (569, 31)


In [7]:
# Encode target variable: M -> 1 (Malignant), B -> 0 (Benign)
le = LabelEncoder()
df['diagnosis_encoded'] = le.fit_transform(df['diagnosis'])

print('Label Encoding:', dict(zip(le.classes_, le.transform(le.classes_))))
print(f'\nTarget Distribution:')
print(df['diagnosis'].value_counts())
print(f'\nMalignant (%): {(df["diagnosis"]=="M").mean()*100:.2f}%')
print(f'Benign    (%): {(df["diagnosis"]=="B").mean()*100:.2f}%')

Label Encoding: {'B': np.int64(0), 'M': np.int64(1)}

Target Distribution:
diagnosis
B    357
M    212
Name: count, dtype: int64

Malignant (%): 37.26%
Benign    (%): 62.74%


## 📊 4. Exploratory Data Analysis (EDA)

In [8]:
# ---- 4.1 Target Class Distribution ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df['diagnosis'].value_counts()
bars = axes[0].bar(counts.index, counts.values,
                   color=[PALETTE['M'], PALETTE['B']], width=0.5, edgecolor='black')
axes[0].set_title('Diagnosis Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Diagnosis')
axes[0].set_ylabel('Count')
for bar, count in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(count), ha='center', va='bottom', fontweight='bold')

axes[1].pie(counts.values, labels=['Malignant', 'Benign'],
            autopct='%1.1f%%', startangle=90,
            colors=[PALETTE['M'], PALETTE['B']],
            explode=(0.05, 0.05), shadow=True)
axes[1].set_title('Diagnosis Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Class distribution plot saved.')

✅ Class distribution plot saved.


In [9]:
# ---- 4.2 Feature Distributions by Diagnosis (Mean Features) ----
mean_features = [col for col in df.columns if '_mean' in col]
print(f'Mean features found: {mean_features}')

n_feats = len(mean_features)
ncols = 3
nrows = (n_feats + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()

for i, feature in enumerate(mean_features):
    for diag, color in PALETTE.items():
        data = df[df['diagnosis'] == diag][feature]
        axes[i].hist(data, bins=25, alpha=0.6, color=color, label=diag, edgecolor='white')
    axes[i].set_title(feature.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].legend(fontsize=8)

# Hide extra axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions by Diagnosis (Mean Features)',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature distribution plots saved.')

Mean features found: ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean']


✅ Feature distribution plots saved.


In [10]:
# ---- 4.3 Boxplots for Mean Features ----
n_feats = len(mean_features)
ncols = 3
nrows = (n_feats + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, nrows * 4))
axes = axes.flatten()

for i, feature in enumerate(mean_features):
    sns.boxplot(
        data=df, x='diagnosis', y=feature,
        palette=PALETTE, ax=axes[i], width=0.5
    )
    axes[i].set_title(feature.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Diagnosis')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Boxplots of Mean Features by Diagnosis',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Boxplot saved.')

✅ Boxplot saved.


In [11]:
# ---- 4.4 Correlation Heatmap ----
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(22, 18))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=False,
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
plt.title('Feature Correlation Heatmap', fontsize=18, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Correlation heatmap saved.')

✅ Correlation heatmap saved.


In [12]:
# ---- 4.5 Top Correlated Features with Target ----
target_corr = numeric_df.corr()['diagnosis_encoded'].drop('diagnosis_encoded')
target_corr_sorted = target_corr.abs().sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 7))
colors = ['#E74C3C' if v > 0 else '#3498DB'
          for v in target_corr[target_corr_sorted.index]]
plt.barh(target_corr_sorted.index[::-1],
         target_corr[target_corr_sorted.index[::-1]],
         color=colors[::-1], edgecolor='black', height=0.6)
plt.axvline(x=0, color='black', linewidth=1)
plt.title('Top 15 Features Correlated with Diagnosis', fontsize=14, fontweight='bold')
plt.xlabel('Pearson Correlation Coefficient')
plt.tight_layout()
plt.savefig('top_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 features correlated with Diagnosis:')
print(target_corr_sorted.head(10).to_string())

Top 10 features correlated with Diagnosis:
concave points_worst    0.793566
perimeter_worst         0.782914
concave points_mean     0.776614
radius_worst            0.776454
perimeter_mean          0.742636
area_worst              0.733825
radius_mean             0.730029
area_mean               0.708984
concavity_mean          0.696360
concavity_worst         0.659610


In [13]:
# ---- 4.6 Pairplot (Top 5 Features) ----
top5_features = list(target_corr_sorted.head(5).index) + ['diagnosis']
pair_df = df[top5_features].copy()

pairplot = sns.pairplot(
    pair_df, hue='diagnosis',
    palette=PALETTE, diag_kind='kde',
    plot_kws={'alpha': 0.6}, height=2.5
)
pairplot.fig.suptitle('Pairplot of Top 5 Features', y=1.02,
                       fontsize=14, fontweight='bold')
plt.savefig('pairplot.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Pairplot saved.')

✅ Pairplot saved.


In [14]:
# ---- 4.7 Violin Plots ----
top6 = list(target_corr_sorted.head(6).index)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(top6):
    sns.violinplot(
        data=df, x='diagnosis', y=feat,
        palette=PALETTE, ax=axes[i], inner='quart'
    )
    axes[i].set_title(feat.replace('_', ' ').title(), fontweight='bold')

plt.suptitle('Violin Plots — Top 6 Predictive Features',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('violin_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Violin plots saved.')

✅ Violin plots saved.


## 🔧 5. Feature Engineering & Data Preparation

In [15]:
# Separate features and target
feature_cols = [col for col in df.columns
                if col not in ['diagnosis', 'diagnosis_encoded']]

X = df[feature_cols]
y = df['diagnosis_encoded']

print(f'Features shape  : {X.shape}')
print(f'Target shape    : {y.shape}')
print(f'\nFeature columns ({len(feature_cols)}):')
print(feature_cols)

Features shape  : (569, 30)
Target shape    : (569,)

Feature columns (30):
['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']


In [16]:
# Train-Test Split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape} | Labels: {dict(y_train.value_counts())}')
print(f'Test set     : {X_test.shape}  | Labels: {dict(y_test.value_counts())}')

Training set : (455, 30) | Labels: {0: np.int64(285), 1: np.int64(170)}
Test set     : (114, 30)  | Labels: {0: np.int64(72), 1: np.int64(42)}


In [17]:
# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('✅ Features scaled using StandardScaler')
print(f'   Train mean ≈ {X_train_scaled.mean():.4f} (should be ~0)')
print(f'   Train std  ≈ {X_train_scaled.std():.4f}  (should be ~1)')

✅ Features scaled using StandardScaler
   Train mean ≈ 0.0000 (should be ~0)
   Train std  ≈ 1.0000  (should be ~1)


## 🔬 6. PCA — Dimensionality Reduction & Visualization

In [18]:
# PCA for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(scaler.transform(X))

pca_df = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'Diagnosis': df['diagnosis'].values
})

explained_var = pca.explained_variance_ratio_ * 100
print(f'PC1 explains: {explained_var[0]:.2f}% of variance')
print(f'PC2 explains: {explained_var[1]:.2f}% of variance')
print(f'Total: {sum(explained_var):.2f}% variance explained by 2 components')

PC1 explains: 43.65% of variance
PC2 explains: 19.20% of variance
Total: 62.85% variance explained by 2 components


In [19]:
# PCA Scatter Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 2D PCA scatter
for diag, color in PALETTE.items():
    mask = pca_df['Diagnosis'] == diag
    label = 'Malignant' if diag == 'M' else 'Benign'
    axes[0].scatter(pca_df[mask]['PC1'], pca_df[mask]['PC2'],
                    c=color, label=label, alpha=0.7, s=50, edgecolors='white', lw=0.5)
axes[0].set_xlabel(f'PC1 ({explained_var[0]:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({explained_var[1]:.1f}% variance)')
axes[0].set_title('PCA — 2D Projection of Dataset', fontweight='bold')
axes[0].legend()

# Explained variance cumulative
pca_full = PCA(random_state=42).fit(scaler.transform(X))
cum_var = np.cumsum(pca_full.explained_variance_ratio_ * 100)
axes[1].plot(range(1, len(cum_var)+1), cum_var, 'b-o', markersize=4)
axes[1].axhline(y=95, color='r', linestyle='--', label='95% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance (%)')
axes[1].set_title('PCA — Cumulative Explained Variance', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('pca_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ PCA analysis plot saved.')

✅ PCA analysis plot saved.


## 🤖 7. Model Training & Evaluation

In [20]:
# Define models
models = {
    'Logistic Regression'    : LogisticRegression(max_iter=5000, random_state=42),
    'K-Nearest Neighbors'    : KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes'            : GaussianNB(),
    'Decision Tree'          : DecisionTreeClassifier(random_state=42),
    'Random Forest'          : RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting'      : GradientBoostingClassifier(n_estimators=200, random_state=42),
    'AdaBoost'               : AdaBoostClassifier(n_estimators=200, random_state=42),
    'Support Vector Machine' : SVC(probability=True, random_state=42),
}

# Cross-validation scores (5-fold stratified)
print('📊 Cross-Validation Accuracy (5-Fold Stratified):')
print('-' * 55)

cv_results = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train,
                                cv=skf, scoring='accuracy')
    cv_results[name] = cv_scores
    print(f'{name:<30} Mean: {cv_scores.mean():.4f}  Std: {cv_scores.std():.4f}')

print('-' * 55)

📊 Cross-Validation Accuracy (5-Fold Stratified):
-------------------------------------------------------
Logistic Regression            Mean: 0.9736  Std: 0.0149
K-Nearest Neighbors            Mean: 0.9670  Std: 0.0155
Naive Bayes                    Mean: 0.9385  Std: 0.0256
Decision Tree                  Mean: 0.9231  Std: 0.0326


Random Forest                  Mean: 0.9626  Std: 0.0149


Gradient Boosting              Mean: 0.9692  Std: 0.0128


AdaBoost                       Mean: 0.9714  Std: 0.0149
Support Vector Machine         Mean: 0.9714  Std: 0.0054
-------------------------------------------------------


In [21]:
# Train all models on full training set and evaluate on test set
results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    results[name] = {
        'model'     : model,
        'accuracy'  : accuracy_score(y_test, y_pred),
        'precision' : precision_score(y_test, y_pred, zero_division=0),
        'recall'    : recall_score(y_test, y_pred, zero_division=0),
        'f1'        : f1_score(y_test, y_pred, zero_division=0),
        'roc_auc'   : roc_auc_score(y_test, y_prob),
        'mcc'       : matthews_corrcoef(y_test, y_pred),
        'y_pred'    : y_pred,
        'y_prob'    : y_prob,
    }

# Results DataFrame
results_df = pd.DataFrame([
    {
        'Model'     : name,
        'Accuracy'  : f"{v['accuracy']*100:.2f}%",
        'Precision' : f"{v['precision']*100:.2f}%",
        'Recall'    : f"{v['recall']*100:.2f}%",
        'F1-Score'  : f"{v['f1']*100:.2f}%",
        'ROC-AUC'   : f"{v['roc_auc']:.4f}",
        'MCC'       : f"{v['mcc']:.4f}",
    }
    for name, v in results.items()
]).sort_values('Accuracy', ascending=False).reset_index(drop=True)

print('\n📋 Model Performance Summary on Test Set:')
print(results_df.to_string())


📋 Model Performance Summary on Test Set:
                    Model Accuracy Precision  Recall F1-Score ROC-AUC     MCC
0                AdaBoost   97.37%   100.00%  92.86%   96.30%  0.9871  0.9442
1  Support Vector Machine   97.37%   100.00%  92.86%   96.30%  0.9947  0.9442
2     Logistic Regression   96.49%    97.50%  92.86%   95.12%  0.9960  0.9245
3           Random Forest   96.49%   100.00%  90.48%   95.00%  0.9942  0.9258
4       Gradient Boosting   96.49%   100.00%  90.48%   95.00%  0.9954  0.9258
5     K-Nearest Neighbors   95.61%    97.44%  90.48%   93.83%  0.9823  0.9058
6           Decision Tree   92.98%    90.48%  90.48%   90.48%  0.9246  0.8492
7             Naive Bayes   92.11%    92.31%  85.71%   88.89%  0.9891  0.8292


In [22]:
# ---- Model Comparison Bar Chart ----
metrics_plot = pd.DataFrame([
    {
        'Model'     : name,
        'Accuracy'  : v['accuracy'],
        'Precision' : v['precision'],
        'Recall'    : v['recall'],
        'F1-Score'  : v['f1'],
        'ROC-AUC'   : v['roc_auc'],
    }
    for name, v in results.items()
])

metrics_melt = metrics_plot.melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(16, 7))
sns.barplot(data=metrics_melt, x='Model', y='Score', hue='Metric',
            palette='tab10', width=0.7)
plt.title('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Model')
plt.ylabel('Score')
plt.xticks(rotation=30, ha='right')
plt.ylim(0.7, 1.02)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Model comparison chart saved.')

✅ Model comparison chart saved.


In [23]:
# ---- ROC Curves for All Models ----
plt.figure(figsize=(12, 8))

colors = plt.cm.tab10.colors
for i, (name, v) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, v['y_prob'])
    roc_auc = v['roc_auc']
    plt.plot(fpr, tpr, color=colors[i % 10], lw=2,
             label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
plt.xlim([-0.01, 1.01])
plt.ylim([-0.01, 1.01])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ ROC curves saved.')

✅ ROC curves saved.


In [24]:
# ---- Confusion Matrices (all 8 models) ----
n_models = len(results)
ncols = 4
nrows = (n_models + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.5, nrows * 4.5))
axes = axes.flatten()

for i, (name, v) in enumerate(results.items()):
    cm = confusion_matrix(y_test, v['y_pred'])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Benign', 'Malignant'],
        yticklabels=['Benign', 'Malignant'],
        ax=axes[i], cbar=False
    )
    acc = v['accuracy'] * 100
    axes[i].set_title(f'{name}\nAcc: {acc:.1f}%', fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Confusion Matrices — All Models', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Confusion matrices saved.')

✅ Confusion matrices saved.


## 🏆 8. Best Model — Hyperparameter Tuning

In [25]:
# Identify best model by ROC-AUC
best_name = max(results, key=lambda k: results[k]['roc_auc'])
print(f'🏆 Best Model by ROC-AUC: {best_name}')
print(f'   Accuracy : {results[best_name]["accuracy"]*100:.2f}%')
print(f'   ROC-AUC  : {results[best_name]["roc_auc"]:.4f}')
print(f'   F1-Score : {results[best_name]["f1"]*100:.2f}%')

🏆 Best Model by ROC-AUC: Logistic Regression
   Accuracy : 96.49%
   ROC-AUC  : 0.9960
   F1-Score : 95.12%


In [26]:
# Hyperparameter Tuning — Random Forest (GridSearchCV)
print('⚙️  Tuning Random Forest with GridSearchCV...')

rf_param_grid = {
    'n_estimators'      : [100, 200],
    'max_depth'         : [None, 10, 20],
    'min_samples_split' : [2, 5],
    'min_samples_leaf'  : [1, 2],
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_param_grid,
    cv=5, scoring='roc_auc', n_jobs=-1, verbose=0
)
rf_grid.fit(X_train_scaled, y_train)

print(f'✅ Best Parameters : {rf_grid.best_params_}')
print(f'   Best CV ROC-AUC: {rf_grid.best_score_:.4f}')

⚙️  Tuning Random Forest with GridSearchCV...


✅ Best Parameters : {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
   Best CV ROC-AUC: 0.9906


In [27]:
# Evaluate tuned model
best_rf    = rf_grid.best_estimator_
y_pred_best = best_rf.predict(X_test_scaled)
y_prob_best = best_rf.predict_proba(X_test_scaled)[:, 1]

print('📊 Tuned Random Forest — Test Set Performance:')
print(f'   Accuracy  : {accuracy_score(y_test, y_pred_best)*100:.2f}%')
print(f'   Precision : {precision_score(y_test, y_pred_best)*100:.2f}%')
print(f'   Recall    : {recall_score(y_test, y_pred_best)*100:.2f}%')
print(f'   F1-Score  : {f1_score(y_test, y_pred_best)*100:.2f}%')
print(f'   ROC-AUC   : {roc_auc_score(y_test, y_prob_best):.4f}')
print(f'   MCC       : {matthews_corrcoef(y_test, y_pred_best):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_best, target_names=['Benign', 'Malignant']))

📊 Tuned Random Forest — Test Set Performance:
   Accuracy  : 97.37%
   Precision : 100.00%
   Recall    : 92.86%
   F1-Score  : 96.30%
   ROC-AUC   : 0.9950
   MCC       : 0.9442

Classification Report:
              precision    recall  f1-score   support

      Benign       0.96      1.00      0.98        72
   Malignant       1.00      0.93      0.96        42

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114



In [28]:
# ---- Feature Importance (Tuned Random Forest) ----
importances = best_rf.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature'   : feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=False).head(20)

plt.figure(figsize=(12, 8))
cmap = plt.cm.RdYlGn
norm_vals = feat_imp_df['Importance'] / feat_imp_df['Importance'].max()
bar_colors = [cmap(x) for x in norm_vals]

plt.barh(feat_imp_df['Feature'][::-1],
         feat_imp_df['Importance'][::-1],
         color=bar_colors[::-1], edgecolor='black', height=0.7)
plt.xlabel('Feature Importance (Gini)', fontsize=12)
plt.title('Top 20 Feature Importances — Tuned Random Forest',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance chart saved.')

✅ Feature importance chart saved.


## 📈 9. Interactive Visualizations (Plotly)

In [29]:
# ---- Interactive 3D PCA Scatter ----
pca3 = PCA(n_components=3, random_state=42)
X_pca3 = pca3.fit_transform(scaler.transform(X))

pca3_df = pd.DataFrame({
    'PC1': X_pca3[:, 0],
    'PC2': X_pca3[:, 1],
    'PC3': X_pca3[:, 2],
    'Diagnosis': df['diagnosis'].map({'M': 'Malignant', 'B': 'Benign'})
})

fig3d = px.scatter_3d(
    pca3_df, x='PC1', y='PC2', z='PC3',
    color='Diagnosis',
    color_discrete_map={'Malignant': '#E74C3C', 'Benign': '#2ECC71'},
    title='3D PCA Projection of Breast Cancer Dataset',
    opacity=0.75,
    symbol='Diagnosis',
    height=600
)
fig3d.update_traces(marker_size=4)
fig3d.show()
print('✅ Interactive 3D PCA rendered.')

✅ Interactive 3D PCA rendered.


In [30]:
# ---- Interactive Model Comparison (Plotly) ----
metrics_plotly = pd.DataFrame([
    {
        'Model'    : name,
        'Accuracy' : round(v['accuracy']  * 100, 2),
        'Precision': round(v['precision'] * 100, 2),
        'Recall'   : round(v['recall']    * 100, 2),
        'F1-Score' : round(v['f1']        * 100, 2),
        'ROC-AUC'  : round(v['roc_auc']   * 100, 2),
    }
    for name, v in results.items()
]).sort_values('Accuracy', ascending=False)

fig_bar = px.bar(
    metrics_plotly.melt(id_vars='Model', var_name='Metric', value_name='Score (%)'),
    x='Model', y='Score (%)', color='Metric',
    barmode='group',
    title='Interactive Model Performance Comparison',
    height=500,
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig_bar.update_layout(xaxis_tickangle=-30)
fig_bar.show()
print('✅ Interactive model comparison rendered.')

✅ Interactive model comparison rendered.


In [31]:
# ---- Interactive Feature Importance ----
fig_imp = px.bar(
    feat_imp_df.sort_values('Importance'),
    x='Importance', y='Feature',
    orientation='h',
    color='Importance',
    color_continuous_scale='RdYlGn',
    title='Top 20 Feature Importances (Tuned Random Forest)',
    height=600
)
fig_imp.show()
print('✅ Interactive feature importance rendered.')

✅ Interactive feature importance rendered.


In [32]:
# ---- Interactive Radar Chart — Top 3 Model Metrics ----
top3_models = list(metrics_plotly.head(3)['Model'])
categories  = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

fig_radar = go.Figure()
radar_colors = ['#E74C3C', '#3498DB', '#2ECC71']

for model_name, color in zip(top3_models, radar_colors):
    row    = metrics_plotly[metrics_plotly['Model'] == model_name].iloc[0]
    values = [row[c] for c in categories]
    values += [values[0]]
    fig_radar.add_trace(go.Scatterpolar(
        r=values,
        theta=categories + [categories[0]],
        fill='toself',
        name=model_name,
        line_color=color,
        opacity=0.7
    ))

fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[80, 100])),
    title='Radar Chart — Top 3 Models Performance',
    height=500
)
fig_radar.show()
print('✅ Interactive radar chart rendered.')

✅ Interactive radar chart rendered.


## 💾 10. Save Best Model & Predict on New Data

In [33]:
# Save best model and scaler
joblib.dump(best_rf, 'breast_cancer_model.pkl')
joblib.dump(scaler,  'scaler.pkl')
print('✅ Model saved : breast_cancer_model.pkl')
print('✅ Scaler saved: scaler.pkl')

✅ Model saved : breast_cancer_model.pkl
✅ Scaler saved: scaler.pkl


In [34]:
def predict_breast_cancer(feature_values,
                           model_path='breast_cancer_model.pkl',
                           scaler_path='scaler.pkl'):
    """
    Predict whether a tumor is Malignant or Benign.

    Parameters
    ----------
    feature_values : list of 30 floats in training-column order.

    Returns
    -------
    dict with prediction, label, and class probabilities.
    """
    loaded_model  = joblib.load(model_path)
    loaded_scaler = joblib.load(scaler_path)

    input_array  = np.array(feature_values).reshape(1, -1)
    input_scaled = loaded_scaler.transform(input_array)

    prediction  = loaded_model.predict(input_scaled)[0]
    probability = loaded_model.predict_proba(input_scaled)[0]

    label = 'Malignant 🔴' if prediction == 1 else 'Benign 🟢'
    return {
        'prediction'     : int(prediction),
        'label'          : label,
        'prob_benign'    : f'{probability[0]*100:.2f}%',
        'prob_malignant' : f'{probability[1]*100:.2f}%',
    }

print('✅ Prediction function defined.')

✅ Prediction function defined.


In [35]:
# Sample predictions on 5 test samples
print('🔬 Sample Predictions on Test Data:')
print('-' * 70)

for idx in [0, 5, 10, 20, 30]:
    sample_features = X_test.iloc[idx].values.tolist()
    result  = predict_breast_cancer(sample_features)
    actual  = 'Malignant' if y_test.iloc[idx] == 1 else 'Benign'
    correct = '✅' if (result['prediction'] == y_test.iloc[idx]) else '❌'
    print(f'Sample {idx+1:>2} | Actual: {actual:<10} | Predicted: {result["label"]:<20} '
          f'| Benign: {result["prob_benign"]:>7} | Malignant: {result["prob_malignant"]:>7} {correct}')

🔬 Sample Predictions on Test Data:
----------------------------------------------------------------------


Sample  1 | Actual: Benign     | Predicted: Benign 🟢             | Benign:  98.67% | Malignant:   1.33% ✅


Sample  6 | Actual: Benign     | Predicted: Benign 🟢             | Benign:  87.18% | Malignant:  12.82% ✅


Sample 11 | Actual: Malignant  | Predicted: Malignant 🔴          | Benign:   2.48% | Malignant:  97.53% ✅
Sample 21 | Actual: Benign     | Predicted: Benign 🟢             | Benign:  99.73% | Malignant:   0.27% ✅


Sample 31 | Actual: Malignant  | Predicted: Malignant 🔴          | Benign:   0.00% | Malignant: 100.00% ✅


## 📋 11. Final Summary

In [36]:
print('=' * 65)
print('        BREAST CANCER ANALYSIS — PROJECT SUMMARY')
print('=' * 65)
print(f'  Dataset Samples   : {len(df)}')
print(f'  Features Used     : {len(feature_cols)}')
print(f'  Malignant Cases   : {(df["diagnosis"]=="M").sum()} ({(df["diagnosis"]=="M").mean()*100:.1f}%)')
print(f'  Benign Cases      : {(df["diagnosis"]=="B").sum()} ({(df["diagnosis"]=="B").mean()*100:.1f}%)')
print(f'  Models Trained    : {len(models)}')
print(f'  Best Tuned Model  : Tuned Random Forest')
print(f'  Test Accuracy     : {accuracy_score(y_test, y_pred_best)*100:.2f}%')
print(f'  Test ROC-AUC      : {roc_auc_score(y_test, y_prob_best):.4f}')
print(f'  Test F1-Score     : {f1_score(y_test, y_pred_best)*100:.2f}%')
print(f'  Model Saved To    : breast_cancer_model.pkl')
print('=' * 65)

# List all saved images
import os
images = [f for f in os.listdir('.') if f.endswith('.png')]
print(f'\n✅ {len(images)} visualization files saved:')
for img in sorted(images):
    size_kb = os.path.getsize(img) / 1024
    print(f'   📊 {img}  ({size_kb:.1f} KB)')

print('\n🎉 Project completed successfully!')

        BREAST CANCER ANALYSIS — PROJECT SUMMARY
  Dataset Samples   : 569
  Features Used     : 30
  Malignant Cases   : 212 (37.3%)
  Benign Cases      : 357 (62.7%)
  Models Trained    : 8
  Best Tuned Model  : Tuned Random Forest
  Test Accuracy     : 97.37%
  Test ROC-AUC      : 0.9950
  Test F1-Score     : 96.30%
  Model Saved To    : breast_cancer_model.pkl

✅ 12 visualization files saved:
   📊 boxplots.png  (248.3 KB)
   📊 class_distribution.png  (76.9 KB)
   📊 confusion_matrices.png  (103.7 KB)
   📊 correlation_heatmap.png  (266.0 KB)
   📊 feature_distributions.png  (214.1 KB)
   📊 feature_importance.png  (96.5 KB)
   📊 model_comparison.png  (100.9 KB)
   📊 pairplot.png  (1053.0 KB)
   📊 pca_analysis.png  (195.9 KB)
   📊 roc_curves.png  (120.2 KB)
   📊 top_correlations.png  (80.2 KB)
   📊 violin_plots.png  (306.3 KB)

🎉 Project completed successfully!
